In [1]:
import torch
from torch import nn

from clean_code.flexible_bitter_llm import FlexibleBitterLLM, Gemma2RotaryEmbedding, IndependentWrapperGater
from clean_code.bitter_llm import RandomGater
torch.serialization.add_safe_globals([nn.modules.sparse.Embedding, FlexibleBitterLLM])

from transformers import AutoTokenizer

import pandas as pd

# Import matplotlib for plotting
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from IPython.display import HTML
import pickle

import numpy as np

import os

In [2]:
# Refactoring the plotting functions to be more modular:
# Helper function: visualize the hard to predict tokens.
def plot_character_intensities(txt, intensities, intensity_to_value_fn=lambda x: x, colorbar_label="intensity"):
    # Get the token logits (cross-entropy loss for each token)    
    # Normalize the logits to a probability-like scale (higher logits = higher probability of being gated)
    # We use softmax-like normalization to get values between 0 and 1
    # Ensure gate_probs and input_text have the same length
    # The first token is not predicted, so we set its intensity to 0
    #     
    # Create HTML with colored text based on probabilities
    colored_text = ""
    colorbar = ""
    
    # Create a colorbar showing the gradient
    for i in range(11):  # 0.0 to 1.0 in steps of 0.1
        intensity = i / 10
        r = min(1.0, intensity)
        b = max(0.0, 1.0 - intensity)
        color = f"rgb({int(r*255)}, 0, {int(b*255)})"
        white = f"rgb(255, 255, 255)"
        colorbar += f'<span style="color:{white}; background-color:{color}; margin-right:2px; padding:0 5px;">{intensity_to_value_fn(intensity):.2f}</span>'
    
    # Add a legend for the colorbar
    colorbar_html = f'''
    <div style="margin-bottom:10px;">
        <div style="font-family:monospace; font-size:12px; margin-bottom:3px;">{colorbar_label}</div>
        <div style="font-family:monospace; font-size:14px;">{colorbar}</div>
    </div>
    '''
    
    # Process the text with colors
    for char, intensity in zip(txt, intensities):
        # Convert probability to color (blue->red)
        r = min(1.0, intensity)  # Red increases with probability
        b = max(0.0, 1.0 - intensity)  # Blue decreases with probability
        color = f"rgb({int(r*255)}, 0, {int(b*255)})"
        # Add the colored character to the output
        colored_text += f'<span style="color:{white}; background-color:{color};">{char}</span>'
    
    # Display the colorbar and colored text
    display(HTML(f'''
    <div>
        {colorbar_html}
        <div style="font-family:monospace; font-size:14px;">{colored_text}</div>
    </div>
    '''))

In [3]:
def plot_train_losses(losses, column_name):
    # Add a smoothed version of the loss curve
    window_size = 20
    smoothed_ar_loss = losses[column_name].rolling(window=window_size).mean()

    plt.figure(figsize=(10, 6))
    plt.plot(losses[column_name], alpha=0.3, color='blue', label=f'{column_name} (raw)')
    plt.plot(smoothed_ar_loss, linewidth=2, color='blue', label=f'{column_name} (smoothed, window={window_size})')
    plt.xlabel('Training Steps')
    plt.ylabel('Loss (nats/token)')
    plt.title(f'{column_name} During Training (Raw and Smoothed)')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [4]:
byte5_tokenizer = AutoTokenizer.from_pretrained("google/byt5-large")

In [5]:
with open("./test_openwebtext_samples.pkl", 'rb') as f:
    my_samples = pickle.load(f)
    my_samples = my_samples["text"]


(4491,
 'CHICAGO -- Size is a nice characteristic to have in hockey, but the Chicago Blackhawks have proven it\'s not everything.\n\nThe Blackhawks have emphasized speed and skill over size, and they\'ve been successful, winning two of the past five Stanley Cup championships (2010 and 2013).\n\nChicago is preparing for its fifth Western Conference Final in the past seven seasons and is considered by some as the favorite to defeat the bigger Anaheim Ducks; Game 1 is Sunday at Honda Center (3 p.m. ET; NBC, CBC, TVA Sports). That probably wouldn\'t have happened a decade ago, when giant men on skates clogged the ice surface, but the NHL and game have changed.\n\nBeing small isn\'t a detriment anymore for guys with great hands and the ability to skate fast.\n\n"I would say if it was like the early 2000s or late \'90s, it seemed like it was a bigger man\'s game and it would be tough for guys [my] size to end up even making the NHL," said 5-foot-11, 177-pound right wing Patrick Kane, who lea

In [22]:

test_string = my_samples[3]
len(test_string), test_string

(1869,
 'Salt Lake City, UT - Over 20 people gathered at a rally, Feb. 27, organized by Utah Against Police Brutality to demand a community controlled police review board.\n\n“We gather here today to demand that Mayor Jackie Bukuspski and the Salt Lake City Council replace the city’s current inept police civilian review board with a democratic, independent, Community Controlled Police Review Board,” said Michael Christensen of UAPB. “This is the beginning of a fight for a properly-funded board of paid civilians with the power to investigate and subpoena police offers for misconduct, create and amend guidelines that regulate how the police are to behave, and can take action without being influenced by the police or the district attorney’s office.”\n\nAttendees of the rally heard from community members who’d lost loved ones to police violence. Among them was Gina Thayne, whose nephew Dylan Taylor was murdered by police in front of his two cousins - her sons.\n\n“Cops protect and serve, b

In [23]:
test_batch = byte5_tokenizer(test_string, return_tensors="pt")["input_ids"]
test_batch = test_batch[:, :1024]
test_batch = test_batch.to("cuda")

In [24]:

from experiment_32.run import seed_to_discount_rate

models = []
discount_rates = []

for seed in range(42, 47):
    models.append(torch.load(f"experiment_32/model_{seed}.pt", weights_only=False))
    discount_rates.append(seed_to_discount_rate[seed])

In [25]:
def plot_gate_probs(model_out):
    gate_probs = model_out["down_gate_probs"]
    gate_probs = gate_probs.detach().cpu().numpy()
    plot_character_intensities(test_string, gate_probs[0], colorbar_label="gate probability")

In [27]:
for model, discount_rate in zip(models, discount_rates):
    print(f"{discount_rate=}")
    with torch.no_grad():
        out = model(test_batch)
    plot_gate_probs(out)

discount_rate=0.8


discount_rate=0.9


discount_rate=0.95


discount_rate=0.97


discount_rate=0.99
